In [1]:
from transformers import AutoTokenizer, RagRetriever, RagSequenceForGeneration, RagTokenForGeneration
import torch

In [2]:
tokenizer = AutoTokenizer.from_pretrained("facebook/rag-sequence-nq")
retriever = RagRetriever.from_pretrained("facebook/rag-sequence-nq", index_name="exact", use_dummy_dataset=True)
model = RagSequenceForGeneration.from_pretrained("facebook/rag-token-nq", retriever=retriever)
print("model loaded")

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'RagTokenizer'. 
The class this function is called from is 'DPRQuestionEncoderTokenizer'.
The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'RagTokenizer'. 
The class this function is called from is 'DPRQuestionEncoderTokenizerFast'.
The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'RagTokenizer'. 
The class this function is called from is 'BartTokenizer'.
The tokenizer class you load from this checkpoint is not the same type as the class this function is called fr

model loaded


In [34]:
inputs = tokenizer("What is Task Decomposition?", return_tensors="pt")
input_ids = inputs["input_ids"]
print(input_ids)

tensor([[  101,  2054,  2003,  4708, 22511,  1029,   102]])


In [35]:
question_hidden_states = model.question_encoder(input_ids)[0]
#question_hidden_states = question_hidden_states.detach().numpy().ravel()
print(question_hidden_states.shape)

torch.Size([1, 768])


In [36]:
docs_dict = retriever(input_ids.numpy(), question_hidden_states.detach().numpy(), return_tensors="pt")
doc_scores = torch.bmm(question_hidden_states.unsqueeze(1), docs_dict["retrieved_doc_embeds"].float().transpose(1, 2)).squeeze(1)
#gen_model = RagTokenForGeneration.from_pretrained("facebook/rag-token-nq", use_dummy_dataset=True)
generated = model.generate(context_input_ids=docs_dict["context_input_ids"], context_attention_mask=docs_dict["context_attention_mask"], doc_scores=doc_scores)
generated_string = tokenizer.batch_decode(generated, skip_special_tokens=True)
print(generated_string)


[' is a random variable']


In [6]:
print("done")

done


In [9]:
import numpy as np
documents = ['all glitters are not gold', 'an apple a day keep the doctor away', 'John is an mbbs doctor']
query = 'apple'
rag = []
for i in range(len(documents)):
    inputs = tokenizer(documents[i], return_tensors="pt")
    input_ids = inputs["input_ids"]
    question_hidden_states = model.question_encoder(input_ids)[0]
    question_hidden_states = question_hidden_states.detach().numpy().ravel()
    rag.append(question_hidden_states)
rag = np.asarray(rag) 
print(rag.shape)

(3, 768)


In [16]:
from numpy import dot
from numpy.linalg import norm
query = "modi"
inputs = tokenizer(query, return_tensors="pt")
input_ids = inputs["input_ids"]
query = model.question_encoder(input_ids)[0]
query = query.detach().numpy().ravel()
print(query.shape)
index = -1
accuracy = 0
for i in range(len(rag)):
    predict_score = dot(rag[i], query)/(norm(rag[i])*norm(query))
    if predict_score > accuracy:
        accuracy = predict_score
        index = i
print(accuracy)
print(index)

(768,)
0.774115
0


In [29]:
from transformers import pipeline
from transformers import RagTokenizer
tokenizer = RagTokenizer.from_pretrained("facebook/rag-token-nq")
retriever = RagRetriever.from_pretrained("facebook/rag-token-nq", index_name="exact")
model = RagSequenceForGeneration.from_pretrained("facebook/rag-token-nq", retriever=retriever)

# Initialize a pipeline for easier handling of the RAG model
rag_pipeline = pipeline("text2text-generation", model=model, tokenizer=tokenizer)
print("loaded")

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'RagTokenizer'. 
The class this function is called from is 'DPRQuestionEncoderTokenizer'.
The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'RagTokenizer'. 
The class this function is called from is 'DPRQuestionEncoderTokenizerFast'.
The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'RagTokenizer'. 
The class this function is called from is 'BartTokenizer'.
The tokenizer class you load from this checkpoint is not the same type as the class this function is called fr

KeyboardInterrupt: 

In [5]:
from numpy import dot
from numpy.linalg import norm
import numpy as np
import os

In [19]:
query = 'Stop'
documents = []
names = []
search = []
rag = []
for root, dirs, directory in os.walk('RagApp/static/files'):
    for j in range(len(directory)):
        with open(root+"/"+directory[j], "rb") as file:
            data = file.read()
        file.close()
        data = data.decode()
        data = data[0:2600]
        names.append(directory[j])
        inputs = tokenizer(data, return_tensors="pt")
        input_ids = inputs["input_ids"]
        question_hidden_states = model.question_encoder(input_ids)[0]
        question_hidden_states = question_hidden_states.detach().numpy().ravel()
        rag.append(question_hidden_states)
rag = np.asarray(rag)
inputs = tokenizer(query, return_tensors="pt")
input_ids = inputs["input_ids"]
query = model.question_encoder(input_ids)[0]
query = query.detach().numpy().ravel()
for i in range(len(rag)):
    predict_score = dot(rag[i], query)/(norm(rag[i])*norm(query))
    if predict_score > 0.50:
        search.append([names[i], predict_score])
search.sort(key = lambda x : x[1], reverse=True)
print(search)

RuntimeError: The size of tensor a (517) must match the size of tensor b (512) at non-singleton dimension 1